# BNB / NeuralProphet and CatBoost

**Historical exploratory experiment — not a validated benchmark.**

Source workspace filename: `BNB_NeuralProphet.ipynb`. See `../README.md` and `../../docs/source_manifest.json` for status and provenance.

Saved outputs were cleared for publication. Dataset paths were made relative; run from `notebooks/exploratory/`. Model dependencies and data access must be configured separately. Do not quote accuracy from this notebook without reproducing its split, target alignment, and calibration protocol.


In [ ]:
############# neural

In [ ]:
import os
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from binance.client import Client
from neuralprophet import NeuralProphet, set_log_level
from catboost import CatBoostRegressor
from sklearn.model_selection import TimeSeriesSplit
from btc_features import build_features

import neuralprophet
import warnings

set_log_level("ERROR")
warnings.filterwarnings("ignore", category=UserWarning)
os.environ["PYTHONWARNINGS"] = "ignore"

print(f"NeuralProphet version: {neuralprophet.__version__}")

In [ ]:
# ========== FETCH BNB LOW PRICE FROM BINANCE (WITH CACHE) ==========
# Cache raw daily lows to avoid re-hitting the API on each run
DATA_CACHE = Path("../../data/raw/bnb_low_daily.csv")

BINANCE_SYMBOL = "BNBUSDT"
BINANCE_INTERVAL = Client.KLINE_INTERVAL_1DAY
BINANCE_LIMIT = 1000              # Binance max rows per request for klines
BINANCE_SLEEP_SECONDS = 0.25      # polite pause between chunked requests

def fetch_bnb_low_binance(start: str, end: str, max_retries: int = 3) -> pd.DataFrame:
    """
    Download BNB/USDT OHLC data from the Binance API (chunked to respect limits)
    and keep only the LOW column. Returns DataFrame with ['ds', 'y'].
    """
    import time

    client = Client()
    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end)
    chunk_start = start_dt
    all_rows = []

    while chunk_start < end_dt:
        chunk_end = min(chunk_start + timedelta(days=BINANCE_LIMIT - 1), end_dt)
        start_str = chunk_start.strftime("%Y-%m-%d")
        end_str = (chunk_end + timedelta(days=1)).strftime("%Y-%m-%d")
        print(
            f"[INFO] Chunk {start_str} → {chunk_end.date()}: requesting up to {BINANCE_LIMIT} rows",
            flush=True,
        )

        for attempt in range(max_retries):
            try:
                klines = client.get_historical_klines(
                    symbol=BINANCE_SYMBOL,
                    interval=BINANCE_INTERVAL,
                    start_str=start_str,
                    end_str=end_str,
                    limit=BINANCE_LIMIT,
                )
                if not klines:
                    raise RuntimeError("Binance returned empty data.")
                all_rows.extend(klines)
                break
            except Exception as exc:
                error_msg = str(exc).lower()
                wait_time = max(5, (attempt + 1) * 5)
                if any(code in error_msg for code in ["rate limit", "429", "1003", "too many requests"]):
                    wait_time = max(wait_time, 10)
                    print(
                        f"[WARNING] Rate-limit during chunk {start_str} → {chunk_end.date()}, waiting {wait_time}s...",
                    )
                else:
                    print(f"[ERROR] Attempt {attempt + 1} failed: {exc}")
                if attempt == max_retries - 1:
                    raise
                time.sleep(wait_time)
        time.sleep(BINANCE_SLEEP_SECONDS)
        chunk_start = chunk_end + timedelta(days=1)

    if not all_rows:
        raise RuntimeError("Binance returned no rows across all chunks.")

    df_raw = pd.DataFrame(all_rows, columns=[
        "timestamp", "open", "high", "low", "close",
        "volume", "close_time", "quote_asset_volume",
        "number_of_trades", "taker_buy_base_asset_volume",
        "taker_buy_quote_asset_volume", "ignore"
    ])
    df_raw["ds"] = pd.to_datetime(df_raw["timestamp"], unit="ms")
    df_raw["y"] = pd.to_numeric(df_raw["low"], errors="coerce")

    df_clean = (
        df_raw.dropna(subset=["ds", "y"])
        .loc[df_raw["y"] > 0, ["ds", "y"]]
        .sort_values("ds")
        .reset_index(drop=True)
    )
    print(f"[INFO] ✅ Downloaded {len(df_clean)} rows of BNB data from Binance.")
    print(f"[INFO] Coverage: {df_clean['ds'].min().date()} → {df_clean['ds'].max().date()}")
    return df_clean

START = "2017-08-17"
END = datetime.utcnow().strftime("%Y-%m-%d")

if DATA_CACHE.exists():
    df = pd.read_csv(DATA_CACHE, parse_dates=["ds"])
    df["y"] = pd.to_numeric(df["y"], errors="coerce")
    df = df.dropna(subset=["ds", "y"]).sort_values("ds").reset_index(drop=True)
    print(f"[INFO] Loaded {len(df)} rows from cache {DATA_CACHE}.")
else:
    df = fetch_bnb_low_binance(START, END)
    df.to_csv(DATA_CACHE, index=False)
    print(f"[INFO] Saved fresh download to {DATA_CACHE.resolve()}")

print(f"\n[INFO] Data source: Binance API (BNBUSDT LOW) | Rows: {len(df)}")
print(f"[INFO] Stats → Min={df['y'].min():.2f}, Max={df['y'].max():.2f}, Mean={df['y'].mean():.2f}")
df.head(10)

In [ ]:
# ========== FEATURE ENGINEERING & DATA CONSISTENCY ==========
def engineer_features(price_df: pd.DataFrame) -> pd.DataFrame:
    """Create lagged technical indicators to help NeuralProphet capture dynamics."""
    df_feat = (
        price_df
        .drop_duplicates(subset="ds")
        .sort_values("ds")
        .reset_index(drop=True)
    )

    # Ensure a continuous daily index and forward-fill gaps
    df_feat = (
        df_feat.set_index("ds")
        .resample("D")
        .ffill()
        .reset_index()
    )

    df_feat["returns_pct"] = df_feat["y"].pct_change()
    df_feat["log_return"] = np.log(df_feat["y"]).diff()
    df_feat["volatility_7d"] = df_feat["returns_pct"].rolling(window=7, min_periods=2).std()
    df_feat["sma_14"] = df_feat["y"].rolling(window=14, min_periods=1).mean()
    df_feat["sma_30"] = df_feat["y"].rolling(window=30, min_periods=1).mean()
    df_feat["sma_ratio_14"] = df_feat["y"] / df_feat["sma_14"]
    df_feat["sma_ratio_30"] = df_feat["y"] / df_feat["sma_30"]

    feature_cols = [
        "returns_pct",
        "log_return",
        "volatility_7d",
        "sma_ratio_14",
        "sma_ratio_30",
    ]

    df_feat[feature_cols] = (
        df_feat[feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(method="bfill")
        .fillna(method="ffill")
        .fillna(0.0)
    )
    df_feat[feature_cols] = df_feat[feature_cols].clip(lower=-5, upper=5)

    return df_feat[["ds", "y"] + feature_cols].dropna(subset=["y"]).reset_index(drop=True)

df = engineer_features(df)
FEATURE_COLS = [
    "returns_pct",
    "log_return",
    "volatility_7d",
    "sma_ratio_14",
    "sma_ratio_30",
]
df.head(5)[["ds", "y"] + FEATURE_COLS]

In [ ]:
df[["y"] + FEATURE_COLS].describe().transpose()

In [ ]:
df[FEATURE_COLS + ["y"]].isna().sum()

In [ ]:
# ========== TRAIN / VALIDATION SPLIT CONFIGURATION ==========
VAL_HORIZON = 180  # days kept aside for validation
FORECAST_HORIZON = 60  # days to forecast ahead

if len(df) <= VAL_HORIZON:
    raise ValueError("Validation horizon is larger than the dataset length.")

cols = ["ds", "y", *FEATURE_COLS]

df_slim = df.loc[:, cols].copy()
df_train = df_slim.iloc[:-VAL_HORIZON].reset_index(drop=True)
df_val = df_slim.iloc[-VAL_HORIZON:].reset_index(drop=True)

# Keep only the slimmed columns for downstream use
df = df_slim.reset_index(drop=True)

print(f"[INFO] Training rows: {len(df_train)}")
print(f"[INFO] Validation rows: {len(df_val)}")
print(f"[INFO] Forecast horizon: {FORECAST_HORIZON} days")

In [ ]:
# ========== MODEL CONFIGURATION & TRAINING ==========
# Hyperparameters grouped for easier tuning. Values below are a refined version
# of the original config, aiming for more stable training and better generalization.
HPARAMS = {
    "n_lags": 120,                # was 90
    "n_forecasts": FORECAST_HORIZON,
    "changepoints_range": 0.90,   # was 0.95
    "n_changepoints": 80,         # was 60
    "seasonality_mode": "multiplicative",
    "daily_seasonality": False,
    "weekly_seasonality": True,
    "yearly_seasonality": True,
    "learning_rate": 0.005,       # was 0.01
    "batch_size": 64,             # was 128
    "epochs": 400,                # was 250
    "ar_reg": 0.2,                # was 0.1
    "seasonality_reg": 1.0,       # was 0.5
    "trend_reg": 0.05,            # was 0.1
    "normalize": "minmax",
    "loss_func": "Huber",
}

model = NeuralProphet(**HPARAMS)
model.add_lagged_regressor(names=FEATURE_COLS)

train_metrics = model.fit(df_train, freq="D", validation_df=df_val)
train_metrics.tail()


In [ ]:
# ========== VALIDATION METRICS ==========
val_forecast = model.predict(df_val)
val_results = val_forecast[["ds", "y", "yhat1"]].dropna()

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(y_true - y_pred))))

def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.where(y_true == 0, np.nan, np.abs(y_true))
    return float(np.nanmean(np.abs((y_true - y_pred) / denom)) * 100)

val_mae = float(np.mean(np.abs(val_results["y"] - val_results["yhat1"])))
val_rmse = rmse(val_results["y"].to_numpy(), val_results["yhat1"].to_numpy())
val_mape = mape(val_results["y"].to_numpy(), val_results["yhat1"].to_numpy())
direction_df = val_results[["y", "yhat1"]].diff().dropna()
if direction_df.empty:
    val_direction = float("nan")
else:
    val_direction = float(np.mean(np.sign(direction_df["y"]) == np.sign(direction_df["yhat1"])))

print(f"Validation MAE:  {val_mae:,.2f}")
print(f"Validation RMSE: {val_rmse:,.2f}")
print(f"Validation MAPE: {val_mape:,.2f}%")
if np.isnan(val_direction):
    print("Directional accuracy: N/A (not enough validation points)")
else:
    print(f"Directional accuracy: {val_direction * 100:,.2f}%")

recent_horizon = min(FORECAST_HORIZON, len(val_results))
print(f"\n[INFO] Last {recent_horizon} validation points:")
val_results.tail(recent_horizon)

In [ ]:
# ========== GENERATE FORECAST ==========
future_df = model.make_future_dataframe(
    df,
    periods=FORECAST_HORIZON,
    n_historic_predictions=True,
 )
forecast = model.predict(future_df)
forecast.tail(FORECAST_HORIZON)

In [ ]:
model.plot_latest_forecast(forecast, plotting_backend='matplotlib')

In [ ]:
import matplotlib.pyplot as plt

history_window = 240
pred_col = next((c for c in forecast.columns if c.startswith("yhat")), None)
if pred_col is None:
    raise KeyError("No forecast column starting with 'yhat' was found.")

df_plot = forecast.tail(history_window + FORECAST_HORIZON).copy()
last_actual_date = df["ds"].max()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df_plot["ds"], df_plot["y"], label="Actual", color="#1f77b4")
ax.plot(df_plot["ds"], df_plot[pred_col], label=f"Forecast ({pred_col})", color="#ff7f0e")

lower_col = next((c for c in forecast.columns if c.startswith(f"{pred_col}_lower")), None)
upper_col = next((c for c in forecast.columns if c.startswith(f"{pred_col}_upper")), None)

if lower_col and upper_col and lower_col in df_plot.columns and upper_col in df_plot.columns:
    ax.fill_between(df_plot["ds"], df_plot[lower_col], df_plot[upper_col], alpha=0.2, color="#ff7f0e")

ax.axvline(last_actual_date, color="black", linestyle="--", alpha=0.7, label="Forecast start")
ax.set_title("BNB LOW Price Forecast (History vs Forecast)")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ========== 7-DAY FORECAST SNAPSHOT ==========
from datetime import timedelta

if "forecast" not in globals():
    print("Chưa có biến `forecast`. Hãy chạy lại cell dự báo.")
else:
    pred_col = next((c for c in forecast.columns if c.startswith("yhat")), None)
    if pred_col is None:
        print("Không tìm thấy cột dự báo (yhat). Kiểm tra lại output của model.predict().")
    else:
        lower_col = next((c for c in forecast.columns if c == f"{pred_col}_lower"), None)
        upper_col = next((c for c in forecast.columns if c == f"{pred_col}_upper"), None)
        cols = ["ds", pred_col]
        if lower_col:
            cols.append(lower_col)
        if upper_col:
            cols.append(upper_col)

        last_history_date = df["ds"].max()
        future_mask = forecast["ds"] > last_history_date
        next7 = forecast.loc[future_mask, cols].head(7).copy()

        if next7.empty:
            print("Không có dữ liệu dự báo tương lai. Hãy tăng FORECAST_HORIZON hoặc kiểm tra lại model.")
        else:
            next7.rename(columns={pred_col: "forecast"}, inplace=True)
            if lower_col:
                next7.rename(columns={lower_col: "lower"}, inplace=True)
            if upper_col:
                next7.rename(columns={upper_col: "upper"}, inplace=True)
            next7["pct_change"] = next7["forecast"].pct_change().fillna(0.0) * 100

            print("\nDự đoán cho 7 ngày tới:")
            print(next7.reset_index(drop=True))

            print("\nChi tiết từng ngày:")
            for _, row in next7.iterrows():
                date = row["ds"].date()
                value = row["forecast"]
                pct = row["pct_change"]
                interval = ""
                if "lower" in row.index and not pd.isna(row["lower"]):
                    if "upper" in row.index and not pd.isna(row["upper"]):
                        interval = f" | [{row['lower']:.2f}, {row['upper']:.2f}]"
                    else:
                        interval = f" | ≥ {row['lower']:.2f}"
                print(f"{date}: {value:,.2f} USD ({pct:+.2f}%){interval}")

In [ ]:
# ========== UPGRADED CATBOOST FEATURE SET ==========
RAW_SOURCE = Path("../../data/raw/bnb_low_daily.csv")
df_boost = pd.read_csv(RAW_SOURCE, parse_dates=["ds"])
df_boost["y"] = pd.to_numeric(df_boost["y"], errors="coerce")
df_boost = (
    df_boost.dropna(subset=["ds", "y"])
    .sort_values("ds")
    .reset_index(drop=True)
    .loc[:, ["ds", "y"]]
 )
volume_col = "volume" if "volume" in df_boost.columns else None

FEATURE_CFG = dict(
    time_col="ds",
    price_col="y",
    volume_col=volume_col,
    horizon=1,
    feature_set="plus",
    target_type="price",
)

X_full, y_full, BOOST_FEATURE_COLS = build_features(
    df_boost,
    dropna=True,
    **FEATURE_CFG,
 )
print(f"[INFO] Gradient-boosting feature matrix shape: {X_full.shape}")
X_full.tail()

In [ ]:
# ========== RANDOM-SEARCH TIME-SERIES CV (CATBOOST) ==========
RANDOM_SEED = 42
TRAIN_DAYS = 1200
HOLDOUT_DAYS = 90
CV_SPLITS = 3
MAX_TRIALS = 4

rng = np.random.default_rng(RANDOM_SEED)

if TRAIN_DAYS > 0 and len(X_full) > TRAIN_DAYS:
    X_cat = X_full.iloc[-TRAIN_DAYS:]
    y_cat = y_full.iloc[-TRAIN_DAYS:]
else:
    X_cat = X_full
    y_cat = y_full

min_required = CV_SPLITS + 5
if len(X_cat) <= min_required:
    raise ValueError(
        f"Need more than {min_required} rows for {CV_SPLITS} splits; got {len(X_cat)}."
)

holdout = min(HOLDOUT_DAYS, max(30, len(X_cat) // 10))
holdout = min(holdout, len(X_cat) - min_required)
holdout = max(0, holdout)

train_X = X_cat if holdout == 0 else X_cat.iloc[:-holdout]
train_y = y_cat if holdout == 0 else y_cat.iloc[:-holdout]
holdout_X = None if holdout == 0 else X_cat.iloc[-holdout:]
holdout_y = None if holdout == 0 else y_cat.iloc[-holdout:]

def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = (np.abs(y_true) + np.abs(y_pred)) + 1e-9
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0)

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mape = float(np.mean(np.abs(y_true - y_pred) / np.clip(np.abs(y_true), 1e-6, None)) * 100.0)
    return {
        "mae": mae,
        "rmse": rmse,
        "mape": mape,
        "smape": smape(y_true, y_pred),
    }

def sample_params(rng: np.random.Generator) -> dict:
    return {
        "depth": int(rng.integers(4, 8)),
        "learning_rate": float(rng.uniform(0.02, 0.07)),
        "l2_leaf_reg": float(rng.uniform(1.0, 8.0)),
        "random_strength": float(rng.uniform(0.8, 2.0)),
        "bagging_temperature": float(rng.uniform(0.1, 1.0)),
        "rsm": float(rng.uniform(0.7, 1.0)),
        "iterations": int(rng.integers(400, 900)),
        "min_data_in_leaf": int(rng.integers(1, 32)),
    }

def fit_catboost(X_train: pd.DataFrame, y_train: pd.Series, params: dict, seed: int, eval_set=None) -> CatBoostRegressor:
    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="MAE",
        depth=params["depth"],
        learning_rate=params["learning_rate"],
        l2_leaf_reg=params["l2_leaf_reg"],
        random_strength=params["random_strength"],
        bagging_temperature=params["bagging_temperature"],
        rsm=params["rsm"],
        iterations=params["iterations"],
        min_data_in_leaf=params["min_data_in_leaf"],
        bootstrap_type="Bayesian",
        od_type="Iter",
        od_wait=80,
        random_seed=seed,
        thread_count=-1,
        verbose=False,
    )
    model.fit(X_train, y_train, eval_set=eval_set, use_best_model=eval_set is not None)
    return model

def run_time_series_cv(X: pd.DataFrame, y: pd.Series, params: dict, splits: int, seed: int):
    splitter = TimeSeriesSplit(n_splits=splits)
    metrics = []
    for fold, (tr_idx, val_idx) in enumerate(splitter.split(X), start=1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = fit_catboost(X_tr, y_tr, params, seed + fold)
        preds = model.predict(X_val)
        fold_metric = compute_metrics(y_val.values, preds)
        fold_metric["fold"] = fold
        metrics.append(fold_metric)
    return metrics

trials = []
for trial in range(1, MAX_TRIALS + 1):
    params = sample_params(rng)
    fold_metrics = run_time_series_cv(train_X, train_y, params, CV_SPLITS, RANDOM_SEED + trial * 7)
    mean_mape = float(np.mean([fm["mape"] for fm in fold_metrics]))
    trials.append({
        "trial": trial,
        "params": params,
        "fold_metrics": fold_metrics,
        "score": mean_mape,
    })
    print(f"Trial {trial}: MAPE={mean_mape:.2f}% with params {params}")

best_trial = min(trials, key=lambda x: x["score"])
best_params = best_trial["params"]
print("\n[INFO] Best CatBoost params:")
print(best_params)
print(f"[INFO] Cross-val MAPE: {best_trial['score']:.2f}%")

pd.DataFrame([
    {
        "trial": t["trial"],
        "score": t["score"],
        "depth": t["params"]["depth"],
        "lr": t["params"]["learning_rate"],
        "iters": t["params"]["iterations"],
    }
    for t in trials
]).sort_values("score").reset_index(drop=True)

In [ ]:
# ========== TRAIN BEST CATBOOST + HOLDOUT EVALUATION ==========
catboost_model = fit_catboost(
    train_X,
    train_y,
    best_params,
    RANDOM_SEED,
    eval_set=(holdout_X, holdout_y) if holdout_X is not None else None,
 )

holdout_metrics = None
if holdout_X is not None:
    holdout_preds = catboost_model.predict(holdout_X)
    holdout_metrics = compute_metrics(holdout_y.values, holdout_preds)
    print("Holdout metrics:")
    for key, value in holdout_metrics.items():
        suffix = "%" if key in {"mape", "smape"} else ""
        print(f"  {key.upper():<6}: {value:.3f}{suffix}")
else:
    print("No holdout window reserved; using all rows for CV.")

catboost_full_model = fit_catboost(X_cat, y_cat, best_params, RANDOM_SEED)
holdout_metrics

In [ ]:
# ========== ITERATIVE FORECAST WITH CATBOOST ==========
def infer_frequency(ds_series: pd.Series) -> pd.Timedelta:
    diffs = ds_series.sort_values().diff().dropna()
    if diffs.empty:
        return pd.Timedelta(days=1)
    median = diffs.median()
    return median if median != pd.Timedelta(0) else pd.Timedelta(days=1)

def iterative_forecast(
    history_df: pd.DataFrame,
    model: CatBoostRegressor,
    feature_cols: list,
    steps: int,
    feature_kwargs: dict,
) -> pd.DataFrame:
    history = history_df[["ds", "y"]].sort_values("ds").reset_index(drop=True)
    freq = infer_frequency(history["ds"])
    temp = history.copy()
    results = []
    for _ in range(steps):
        X_future, _, _ = build_features(temp, dropna=False, **feature_kwargs)
        for col in feature_cols:
            if col not in X_future.columns:
                X_future[col] = 0.0
        row = X_future[feature_cols].iloc[[-1]].copy()
        row = row.ffill(axis=1).bfill(axis=1).fillna(0.0)
        pred = float(model.predict(row)[0])
        pred = max(pred, 1.0)
        next_ds = temp["ds"].iloc[-1] + freq
        results.append({"ds": next_ds, "yhat": pred})
        temp = pd.concat([temp, pd.DataFrame({"ds": [next_ds], "y": [pred]})], ignore_index=True)
    return pd.DataFrame(results)

CAT_FORECAST_STEPS = 60
catboost_forecast = iterative_forecast(
    df_boost,
    catboost_full_model,
    BOOST_FEATURE_COLS,
    CAT_FORECAST_STEPS,
    FEATURE_CFG,
 )
catboost_forecast.head(10)

In [ ]:
# ========== VISUALIZE CATBOOST FORECAST ==========
import matplotlib.pyplot as plt

history_window = 240
plot_history = df_boost.tail(history_window)
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(plot_history["ds"], plot_history["y"], label="Actual", color="#1f77b4")
ax.plot(catboost_forecast["ds"], catboost_forecast["yhat"], label="CatBoost forecast", color="#d62728")
ax.axvline(plot_history["ds"].iloc[-1], color="black", linestyle="--", alpha=0.6, label="Forecast start")
ax.set_title("CatBoost Upgraded BNB LOW Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ========== CATBOOST 7-DAY FORECAST SNAPSHOT ==========
if catboost_forecast.empty:
    print("Chưa có bảng catboost_forecast. Hãy chạy lại cell dự báo nâng cấp.")
else:
    cat_next7 = catboost_forecast.head(7).copy()
    cat_next7["pct_change"] = cat_next7["yhat"].pct_change().fillna(0.0) * 100
    print("Dự đoán CatBoost cho 7 ngày tới:")
    display(cat_next7)
    print("\nChi tiết:")
    for _, row in cat_next7.iterrows():
        print(f"{row['ds'].date()}: {row['yhat']:,.2f} USD ({row['pct_change']:+.2f}%)")